In [1]:
import cirq
import sys
from collections import Counter
import warnings

In [2]:
sys.path.append('../')
import resource_estimation as res

In [3]:
def info_dict(circuit):
    print(f"Moments:{' ' * (10 - len(str(len(circuit))))}{len(circuit)}")
    total = Counter()
    for op in circuit.all_operations():
        if op.gate in cirq.GateFamily(cirq.Rz):
            total += Counter({"Rz": 1})
        elif op.gate in cirq.GateFamily(cirq.PhasedXZGate):
            total += Counter({"PhXZ": 1})
        elif op.gate in cirq.GateFamily(cirq.MeasurementGate):
            total += Counter({"Measurement": 1})
        else:
            total += Counter({str(op.gate): 1})
    for k, v in total.items():
        print(f"{k}{' ' * (16 - len(k))}{v}")

EPS = 1e-3

## Basic Pygridsynth
Reminder of the basic functionality

In [4]:
example_rz_gate = cirq.Rz(rads=.25)
example_phxz_gate = cirq.PhasedXZGate(x_exponent=.25, z_exponent=.35, axis_phase_exponent=.45)
print(f"{str(example_rz_gate)} -> {res.compile_gateset.approx_phxz(example_rz_gate, EPS)}")
print(f"{str(example_phxz_gate)} -> {res.compile_gateset.approx_phxz(example_phxz_gate, EPS)}")

Rz(0.07957747154594767π) -> SHTSHTHTSHTSHTSHTSHTHTSHTSHTHTHTHTSHTSHTHTHTHTSHTHTSHTHTSHTSHTSHTHTSHTSHTSHTHTHTHTHTHTHTSHTHTSHTSHTHTSHTHTSHTHTHTHTHTHTHTSHTSHTSHTHTSHTHTSHTHTHTHTSHTSHTHTHTHTHTHTSHTSHTSHTHTSHTSHTHTHTSHTSHTSHXWWWW
PhXZ(a=0.45,x=0.25,z=0.35) -> HTSHTSHTHTSHTHTHTSHTHTHTSHTHTSHTSHTSHTHTSHTSHTSHTSHTSHTSHTSHTSHTHTHTHTSHTHTHTSHTSHTSHTSHTSHTSHTSHTHTHTHTHTSHTHTHTSHTHTHTSHTSHTHTHTHTHTHTSHTHTHTSHTHTHTHTSHTSHTSHTSHTSHTSHTHTHTSHTHTSHTHSSSWW


## Initialize Example

In [5]:
qubits = 3
U = cirq.testing.random_unitary(dim=2**qubits, random_state=7)
circuit = cirq.Circuit(cirq.MatrixGate(U).on(*cirq.LineQubit.range(qubits)))
circuit

┌                                                                     ┐
      │ 0.385-0.087j -0.242-0.314j -0.212-0.333j  0.052-0.213j -0.124-0.348j│
      │  -0.062+0.076j  0.207+0.438j -0.318-0.045j                          │
      │ 0.232+0.025j  0.152+0.443j -0.087+0.353j  0.201-0.371j -0.004-0.037j│
      │  -0.145+0.237j -0.135-0.153j -0.437-0.323j                          │
      │ 0.126+0.149j  0.03 +0.25j   0.136+0.067j -0.383-0.487j  0.256-0.094j│
      │  -0.297-0.194j  0.28 +0.201j  0.413+0.045j                          │
      │-0.01 +0.348j -0.438+0.149j -0.048+0.187j -0.394+0.339j  0.259-0.163j│
0: ───│  -0.144+0.023j -0.256+0.088j -0.343+0.227j                          │───
      │-0.376+0.514j  0.308+0.218j -0.262-0.207j -0.119+0.015j -0.239+0.161j│
      │   0.156+0.172j  0.231+0.337j -0.133-0.069j                          │
      │ 0.191+0.111j -0.087-0.23j   0.052+0.309j -0.136+0.011j -0.591+0.344j│
      │  -0.323+0.072j -0.295+0.247j  0.202-0.087j                          │
      │ 0.061+0.24j  -0.16 +0.129j  0.546-0.34j   0.047-0.052j -0.335+0.113j│
      │  -0.144-0.314j  0.188-0.308j -0.315+0.054j                          │
      │-0.327-0.089j  0.223-0.225j  0.007+0.168j  0.256+0.128j  0.135+0.005j│
      │  -0.694+0.033j  0.287+0.068j -0.206+0.21j                           │
      └                                                                     ┘
      │
1: ───#2────────────────────────────────────────────────────────────────────────
      │
2: ───#3────────────────────────────────────────────────────────────────────────

## Clifford + Rz and Clifford + PhasedXZ Compilation

In [6]:
cliff_rz = res.compile_gateset.compile_gateset(circuit, res.compile_gateset.clifford_rz_gateset())
cliff_rz

┌──┐                                                                                                                                                                                                                                                                                                                                                                                                                                               ┌──┐
0: ───Rz(-0.001π)─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────X───Rz(0.499π)───X───Rz(-0.067π)───X───Rz(-0.179π)─────X────Z─────────────S───H────────────Rz(0.443π)───H────────────────────────────────────────────────────────────────────────────────────────────────────────────────────@───Z───H───Rz(0.283π)───H───@───Z────────────H───Rz(0.025π)───H───@─────────────Z────────────H───Rz(0.122π)───H───Rz(-0.171π)────────────────────────────────────────────────────────────────────────────────────────X───Rz(-0.427π)───X───Rz(-0.033π)───X───Rz(-0.24π)─────X─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                      │                │                 │                   │                                                                                                                                                                     │                            │                                     │                                                                                                                                                  │                 │                 │                  │
1: ───Rz(-0.859π)───H───Rz(0.248π)───H───────────────@───Rz(-0.253π)───H───Rz(0.112π)───H───X───Z────────────H───Rz(0.104π)───H─────────────────@───Rz(0.031π)───H───Rz(0.235π)───H───@────────────────┼─────────────────@───Rz(0.813π)─────H┼────Rz(0.791π)────H──────────────────────────────────────────@───Rz(-0.164π)───H───Rz(0.358π)───H────────────────@───Rz(-0.051π)───H───Rz(0.191π)───H───Rz(0.437π)───X────────────────────────────┼─────────────────────────────────────X─────────────Rz(0.294π)───H───Rz(0.124π)───H───@─────────────Rz(0.291π)───H───Rz(0.326π)───H───────────────@───Rz(-0.239π)───H───Rz(0.375π)───H───@─────────────────┼─────────────────@───Rz(0.144π)────H┼────Rz(0.689π)───H─────────────────────────────────@───Rz(-0.868π)───H───Rz(0.353π)───H─────────────────@───Rz(-0.998π)───H───Rz(0.072π)───H───Rz(-0.675π)───
                                                     │                                      │                                                   │                                                      │                                     │                                                             │                                                   │                                                                                │                                                                                     │                                                           │                                                        │                                    │                                                   │                                                    │
2: ───Rz(0.63π)─────H───Rz(0.457π)───H───Rz(-0.3π)───X───Rz(-0.736π)───H───Rz(0.592π)───H───@───Rz(0.968π)───H───Rz(0.533π)───H───Rz(-0.425π)───X───Rz(0.557π)───H───Rz(0.981π)───H────────────────────@─────────────────────────────────────@────Rz(-0.873π)───H───Rz(0.232π)───H────────────Rz(0.552π)───X───Rz(-0.481π)───H───Rz(0.223π)───H───Rz(0.519π)───X───Rz(-0.218π)───H───Rz(0.259π)───H───Rz(0.086π)────────────────────────────────X───Rz(0.345π

In [7]:
info_dict(cliff_rz)

Moments:        95
Rz              75
H               60
CNOT            20
Z               5
S               1


In [8]:
cliff_phxz = res.compile_gateset.compile_gateset(circuit, res.compile_gateset.clifford_phxz_gateset())
cliff_phxz

0: ───PhXZ(a=0.501,x=0,z=-0.00122)──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────X───PhXZ(a=0.00107,x=0,z=0.499)───X───PhXZ(a=0.567,x=0,z=-0.0673)───X───PhXZ(a=0.679,x=0,z=-0.179)───X───PhXZ(a=0.5,x=0.443,z=0)─────────────────────────────────────────────────────────────────────────────────────@───PhXZ(a=-0.5,x=0.283,z=4.44e-16)───@───PhXZ(a=0.5,x=0.0255,z=-4.44e-16)───@───PhXZ(a=-0.5,x=0.122,z=0.329)──────────────────────────────────────────────────────────────────────────────X───PhXZ(a=0.927,x=0,z=-0.427)───X───PhXZ(a=0.533,x=0,z=-0.033)───X───PhXZ(a=0.74,x=0,z=-0.24)──────X─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                        │                                 │                                 │                                │                                                                                                               │                                     │                                      │                                                                                                             │                                │                                │                                 │
1: ───PhXZ(a=0.859,x=0.248,z=0)──────@───PhXZ(a=-0.888,x=0.112,z=0.888)─────X───PhXZ(a=-1,x=0.104,z=-0.896)───────@───PhXZ(a=0.0725,x=0.235,z=-0.556)───@─────────────────────────────────┼─────────────────────────────────@───PhXZ(a=0.703,x=0.791,z=0)────┼─────────────────────────────────────@───PhXZ(a=0.867,x=0.358,z=0)────────@───PhXZ(a=0.918,x=0.191,z=-0.481)───X─────────────────────────────────────┼──────────────────────────────────────X───PhXZ(a=-0.294,x=0.124,z=0.935)───@───PhXZ(a=0.35,x=0.326,z=0)────────@───PhXZ(a=0.589,x=0.375,z=-0.51)────@────────────────────────────────┼────────────────────────────────@───PhXZ(a=-0.0644,x=0.689,z=0)───┼─────────────────────────────────────@───PhXZ(a=0.803,x=0.353,z=0)─────────@───PhXZ(a=-0.199,x=0.072,z=-0.476)───
                                     │                                      │                                     │                                                                       │                                                                  │                                     │                                    │                                                                          │                                                                           │                                   │                                                                     │                                                                  │                                     │                                     │
2: ───PhXZ(a=-0.63,x=0.457,z=0.33)───X───PhXZ(a=0.736,x=0.592,z=-0.00785)───@───PhXZ(a=-0.239,x=0.533,z=-0.186)───X───PhXZ(a=-0.557,x=0.981,z=-0.794)─────────────────────────────────────@──────────────────────────────────────────────────────────────────@───PhXZ(a=-0.477,x=0.232,z=-0.972)───X───PhXZ(a=0.481,x=0.223,z=0.0387)───X───PhXZ(a=0.218,x=0.259,z=-0.132)─────────────────────────────────────────X───PhXZ(a=-0.345,x=0.588,z=-0.31)──────────────────────────────────────────X───PhXZ(a=0.48,x=0.202,z=0.0404)───X───PhXZ(a=-0.979,x=0.111,z=0.959)────────────────────────────────────@──────────────────────────────────────────────────────────────────@───PhXZ(a=-0.429,x=0.323,z=-0.833)───X───PhXZ(a=-0.522,x=0.124,z=0.0448)───X───PhXZ(a=-0.063,x=0.109,z=0.61)─────

In [9]:
info_dict(cliff_phxz)

Moments:        41
PhXZ            37
CNOT            20


## Clifford + T Compilation

In [10]:
rz_to_cliff_t = res.compile_gateset.compile_gateset(cliff_rz, res.compile_gateset.clifford_t_gateset(atol=EPS), verbose=False)
phxz_to_cliff_t = res.compile_gateset.compile_gateset(cliff_phxz, res.compile_gateset.clifford_t_gateset(atol=EPS), verbose=False)
cliff_t_direct = res.compile_gateset.compile_gateset(circuit, res.compile_gateset.clifford_t_direct_gateset(eps=EPS), verbose=False)

In [11]:
info_dict(rz_to_cliff_t)

Moments:      3683
X               35
S               1357
H               2550
T               2446
CNOT            20
Z               5


In [12]:
info_dict(phxz_to_cliff_t)

Moments:      3934
X               19
S               1505
T               2850
H               2871
CNOT            20


In [13]:
info_dict(cliff_t_direct)

Moments:      5525
X               22
S               2011
T               3761
H               3812
CNOT            29
